# 19. Dockerizing LLM Applications
**Duration:** 25 min | **Topics:** Containerization, image optimization, secrets

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Skill | Why It Matters |
|---|-------|----------------|
| 1 | Write **multi-stage Dockerfiles** that separate build tools from runtime | Single-stage images ship GCC, make and other tools to prod — bloating size from ~187 MB to ~1.4 GB |
| 2 | Order Docker layers correctly so **iterative builds take seconds not minutes** | Misplaced `COPY . .` before `pip install` invalidates the package cache on every code change |
| 3 | Mount **model weights on a persistent volume** rather than baking them into the image | HuggingFace model weights are often 1–14 GB — they do not belong in a container image |
| 4 | Inject **Azure Key Vault secrets at runtime** (zero secrets in images/git) | A leaked AZURE_OPENAI_KEY hardcoded in a Dockerfile can result in thousands of dollars of abuse |
| 5 | Use **docker-compose** for local development with `.env` files | docker-compose lets you reproduce the exact production environment on any developer laptop |
| 6 | Push images to **Azure Container Registry** without installing Docker locally | ACR Tasks builds the image in the cloud — no Docker Desktop licence needed |

---

## 📐 Architecture

```
Developer laptop
  │  docker-compose up --build
  │  (secrets from .env → never committed)
  ▼
Azure Container Registry  ←  az acr build  (cloud build, no local Docker)
  │
  ▼
Azure Container Apps  →  pulls image, reads AZURE_OPENAI_KEY from Key Vault
```

### Minimum Azure Resources
| Resource | SKU | Cost |
|---|---|---|
| Azure Container Registry | Basic | ~\$0.167/day |
| Azure Key Vault | Standard | ~\$0.03/10 k ops |


In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['docker', 'azure-containerregistry', 'azure-keyvault-secrets', 'azure-identity'])


✅ Ready: docker, azure-containerregistry, azure-keyvault-secrets, azure-identity
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: Understanding Docker Image Layers (Why Multi-Stage Matters)

In [ ]:
# Understand WHY multi-stage builds matter before writing the Dockerfile
# Each FROM instruction = a new stage. Only the LAST stage ships to production.

layer_explanation = {
    "single_stage_image": {
        "layers": [
            "python:3.11 base (~900MB)",
            "apt-get install build tools (gcc, make) (~200MB)",
            "pip install -r requirements.txt (~300MB)",
            "COPY application code (~5MB)",
        ],
        "total_size": "~1.4GB",
        "ships_to_prod": "Everything — including build tools you never need at runtime!"
    },
    "multi_stage_image": {
        "builder_stage": [
            "python:3.11-slim base (~150MB)",
            "pip install --user (packages only, stored in /root/.local)",
        ],
        "runtime_stage": [
            "python:3.11-slim base (~150MB)   ← fresh, no build cruft",
            "COPY --from=builder /root/.local  ← only the installed packages",
            "COPY application code (~5MB)",
        ],
        "total_size": "~187MB",
        "ships_to_prod": "Only runtime packages + your code. Build tools stay in builder."
    }
}

print("Docker Image Size Comparison:")
print("="*55)
for stage, info in layer_explanation.items():
    print(f"\n  {stage.upper()}")
    if "layers" in info:
        for layer in info["layers"]:
            print(f"    + {layer}")
    else:
        print("  Builder (discarded):")
        for l in info.get("builder_stage",[]): print(f"    + {l}")
        print("  Runtime (shipped):")
        for l in info.get("runtime_stage",[]): print(f"    + {l}")
    print(f"  → Total: {info['total_size']}")
    print(f"  → Ships: {info['ships_to_prod']}")


Docker Image Size Comparison:

  SINGLE_STAGE_IMAGE
    + python:3.11 base (~900MB)
    + apt-get install build tools (~200MB)
    + pip install -r requirements.txt (~300MB)
    + COPY application code (~5MB)
  → Total: ~1.4GB
  → Ships: Everything — including build tools you never need at runtime!

  MULTI_STAGE_IMAGE
  Builder (discarded):
    + python:3.11-slim base (~150MB)
    + pip install --user (packages only)
  Runtime (shipped):
    + python:3.11-slim base (~150MB)
    + COPY --from=builder /root/.local
    + COPY application code (~5MB)
  → Total: ~187MB
  → Ships: Only runtime packages + your code. Build tools stay in builder.


## Step 2: Write the Optimized Multi-Stage Dockerfile

## Step 2b: LLM-Specific Layer Ordering & Model Weight Caching

In [ ]:
# LLM-specific Dockerfile patterns
# Key insight: model weights are large (GBs). Layer ordering matters hugely for
# rebuild speed and registry storage costs.

llm_dockerfile = """
# ── LLM-Optimised Dockerfile ─────────────────────────────────────────────────
FROM python:3.11-slim AS builder

# 1. System deps first (changes rarely → cached)
RUN apt-get update && apt-get install -y --no-install-recommends gcc curl \\
    && rm -rf /var/lib/apt/lists/*

# 2. Python deps before copying code (changes occasionally → cached)
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# ── Runtime stage ─────────────────────────────────────────────────────────────
FROM python:3.11-slim AS runtime
RUN addgroup --system app && adduser --system --ingroup app appuser
WORKDIR /app

# 3. Copy installed packages from builder
COPY --from=builder /root/.local /home/appuser/.local

# 4. Copy model config (changes occasionally)
COPY --chown=appuser:app model_config.json .

# 5. Copy application code LAST (changes most often → only this layer rebuilds)
COPY --chown=appuser:app app/ .

# ── LLM-specific env vars ─────────────────────────────────────────────────────
ENV PATH=/home/appuser/.local/bin:$PATH \\
    PYTHONUNBUFFERED=1 \\
    # Cache HuggingFace models to a mounted volume, not inside the container
    TRANSFORMERS_CACHE=/mnt/model-cache \\
    HF_HOME=/mnt/model-cache \\
    # Prevent tokenizer parallelism warnings
    TOKENIZERS_PARALLELISM=false \\
    # Azure OpenAI — injected at runtime via Key Vault, NOT baked in
    AZURE_OPENAI_ENDPOINT="" \\
    AZURE_OPENAI_KEY=""

USER appuser
HEALTHCHECK --interval=30s --timeout=5s CMD curl -f http://localhost:8000/health || exit 1
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
"""

# Explain the key LLM-specific decisions
decisions = [
    ("TRANSFORMERS_CACHE → mounted volume",
     "Model weights (GBs) live on a persistent volume, not inside the image.\n"
     "  Image stays small. Weights survive container restarts. No re-download on scale-out."),
    ("AZURE_OPENAI_KEY left empty",
     "Secret is injected at runtime via Azure Key Vault reference.\n"
     "  Never in the image. Never in git. Never in az containerapp create --env-vars."),
    ("Code copied LAST",
     "Application code changes every deploy. Putting it last means all the\n"
     "  heavy layers (OS, packages, model config) stay cached in ACR."),
    ("--workers 2 in CMD",
     "Uvicorn workers share the same process space → model loaded ONCE per container.\n"
     "  More workers = more RAM used. Tune based on your SKU memory."),
]

print("🐋 LLM-Specific Dockerfile Patterns")
print("=" * 60)
for title, detail in decisions:
    print(f"\n  📌 {title}")
    print(f"     {detail}")

print("\n[SYNTHETIC] Layer sizes after build:")
print("  Base OS (python:3.11-slim) :  130 MB  (cached after first build)")
print("  System dependencies        :   18 MB  (cached)")
print("  Python packages            :  310 MB  (cached until requirements.txt changes)")
print("  model_config.json          :    2 KB  (rarely changes)")
print("  Application code           :   45 KB  ← only this rebuilds on each deploy")
print("  ─────────────────────────────────────")
print("  Total image                :  458 MB  (model weights on volume = not in image)")


🐋 LLM-Specific Dockerfile Patterns

  📌 TRANSFORMERS_CACHE → mounted volume
     Model weights (GBs) live on a persistent volume, not inside the image.
     Image stays small. Weights survive container restarts. No re-download on scale-out.

  📌 AZURE_OPENAI_KEY left empty
     Secret is injected at runtime via Azure Key Vault reference.
     Never in the image. Never in git. Never in az containerapp create --env-vars.

  📌 Code copied LAST
     Application code changes every deploy. Putting it last means all the
     heavy layers (OS, packages, model config) stay cached in ACR.

  📌 --workers 2 in CMD
     Uvicorn workers share the same process space → model loaded ONCE per container.
     More workers = more RAM used. Tune based on your SKU memory.

[SYNTHETIC] Layer sizes after build:
  Base OS (python:3.11-slim) :  130 MB  (cached after first build)
  System dependencies        :   18 MB  (cached)
  Python packages            :  310 MB  (cached until requirements.txt changes)
  mod

## Step 2c: Secrets Management — Azure Key Vault (Not Env Vars)

In [ ]:
# Injecting secrets at runtime via Azure Key Vault (NOT env vars)
# This is the correct production pattern for AZURE_OPENAI_KEY

KV_NAME = 'kv-llm-demo'
RG      = 'rg-llm-demo'
APP     = 'llm-api-app'

# Build each command as a plain string — no nested quotes in f-strings
steps = [
    ('1. Create Key Vault',
     f'az keyvault create --name {KV_NAME} --resource-group {RG} --sku standard'),

    ('2. Store the OpenAI key as a secret',
     f'az keyvault secret set --vault-name {KV_NAME} --name openai-key --value <YOUR_KEY>'),

    ('3. Enable managed identity on Container App',
     f'az containerapp identity assign --name {APP} --resource-group {RG} --system-assigned'),

    ('4. Grant Container App read access to Key Vault',
     'az keyvault set-policy'
     f' --name {KV_NAME}'
     ' --object-id $(az containerapp show'
     f' --name {APP} --resource-group {RG} --query identity.principalId -o tsv)'
     ' --secret-permissions get list'),

    ('5. Reference secret in Container App — never stored in plain text',
     'az containerapp secret set'
     f' --name {APP} --resource-group {RG}'
     f' --secrets openai-key=keyvaultref:{KV_NAME}/secrets/openai-key'),

    ('6. Expose as env var inside the container',
     'az containerapp update'
     f' --name {APP} --resource-group {RG}'
     ' --set-env-vars AZURE_OPENAI_KEY=secretref:openai-key'),
]

print('=' * 70)
print('  Azure Key Vault Secret Injection — Zero Secrets in Images')
print('=' * 70)
for title, cmd in steps:
    print(f'\n--- {title} ---')
    print(cmd)

print('\n[SYNTHETIC DEMO OUTPUT]')
print(f"Key Vault '{KV_NAME}' created")
print("Secret 'openai-key' stored (not visible in portal by default)")
print(f"Managed identity assigned to {APP}")
print("Key Vault policy granted — app can GET/LIST secrets")
print(f"Container now reads AZURE_OPENAI_KEY from Key Vault at startup")
print()
print('✅ Secret never appears in: Dockerfile, image layers, git, az CLI history')
print()
print('Why this matters:')
print('  • Leaked API keys in public GitHub repos are found by bots within minutes')
print('  • Key Vault gives you audit logs: who accessed the key and when')
print('  • Rotating a compromised key = one Key Vault operation, no redeploy needed')


  Azure Key Vault Secret Injection — Zero Secrets in Images

--- 1. Create Key Vault ---
az keyvault create --name kv-llm-demo --resource-group rg-llm-demo --sku standard

--- 2. Store the OpenAI key as a secret ---
az keyvault secret set --vault-name kv-llm-demo --name openai-key --value <YOUR_KEY>

--- 3. Enable managed identity on Container App ---
az containerapp identity assign --name llm-api-app --resource-group rg-llm-demo --system-assigned

--- 4. Grant Container App read access to Key Vault ---
az keyvault set-policy --name kv-llm-demo --object-id <principal-id> --secret-permissions get list

--- 5. Reference secret in Container App ---
az containerapp secret set --name llm-api-app ... --secrets openai-key=keyvaultref:kv-llm-demo/secrets/openai-key

--- 6. Expose as env var inside the container ---
az containerapp update --name llm-api-app ... --set-env-vars AZURE_OPENAI_KEY=secretref:openai-key

[SYNTHETIC DEMO OUTPUT]
Key Vault 'kv-llm-demo' created
Secret 'openai-key' store

In [ ]:
dockerfile = """# ── Stage 1: Builder ─────────────────────────────────────────────
FROM python:3.11-slim AS builder

# Security: create non-root user before doing anything else
RUN addgroup --system appgroup && \
    adduser --system --ingroup appgroup --no-create-home appuser

WORKDIR /app

# Layer caching: copy requirements FIRST, code SECOND
# Reason: requirements change rarely, code changes often.
# Docker caches each layer — if requirements.txt unchanged, pip install is skipped.
COPY requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# ── Stage 2: Runtime (this is what gets shipped) ─────────────────
FROM python:3.11-slim AS runtime

# Recreate user in runtime stage
RUN addgroup --system appgroup && \
    adduser --system --ingroup appgroup --no-create-home appuser

# Minimal runtime deps only (curl for HEALTHCHECK)
RUN apt-get update && apt-get install -y --no-install-recommends curl && \
    rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Copy ONLY installed packages from builder — NOT the build tools
COPY --from=builder /root/.local /home/appuser/.local
COPY --chown=appuser:appgroup . .

USER appuser

ENV PATH=/home/appuser/.local/bin:$PATH \
    PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1

# Health check: Container Apps and ACA use this to decide if app is ready
HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
"""

requirements = """fastapi==0.111.0
uvicorn[standard]==0.29.0
pydantic==2.7.0
openai==1.30.0
httpx==0.27.0
python-dotenv==1.0.1
python-jose[cryptography]==3.3.0
"""

with open("Dockerfile", "w") as f: f.write(dockerfile)
with open("requirements.txt", "w") as f: f.write(requirements)
print("✅ Dockerfile written")
print("\nKey layer caching insight:")
print("  COPY requirements.txt .   ← cached if requirements unchanged")
print("  RUN pip install ...        ← skipped if above layer cached")
print("  COPY . .                   ← only this re-runs on code changes")
print("\nResult: iterative builds take ~5s instead of ~3min")


✅ Dockerfile written

Key layer caching insight:
  COPY requirements.txt .   ← cached if requirements unchanged
  RUN pip install ...        ← skipped if above layer cached
  COPY . .                   ← only this re-runs on code changes

Result: iterative builds take ~5s instead of ~3min


## Step 3: docker-compose for Local Development

In [ ]:
# docker-compose lets you run the container locally with proper secret injection
# NEVER put real secrets in docker-compose.yml — use .env file (gitignored)

compose_yaml = """
version: "3.9"

services:
  llm-api:
    build:
      context: .
      dockerfile: Dockerfile
      target: runtime          # Use the runtime stage, not builder
    ports:
      - "8000:8000"
    environment:
      # Secrets come from .env file — NEVER hardcoded here
      - AZURE_OPENAI_KEY=${AZURE_OPENAI_KEY}
      - AZURE_OPENAI_ENDPOINT=${AZURE_OPENAI_ENDPOINT}
      - AZURE_OPENAI_API_VERSION=2024-02-01
      - MODEL_NAME=gpt-4o-mini
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 5s
      retries: 3
    restart: unless-stopped

  # Optional: local Redis for caching LLM responses
  redis:
    image: redis:7-alpine
    ports: ["6379:6379"]
"""

env_example = """# .env.example — copy to .env and fill in real values
# .env is in .gitignore — NEVER commit real keys!
AZURE_OPENAI_KEY=your-key-here
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
"""

dockerignore = """__pycache__/
*.pyc
.env
.env.*
*.key  *.pem  *.p12
.git/
*.ipynb
tests/
.venv/
"""

with open("docker-compose.yml","w") as f: f.write(compose_yaml)
with open(".env.example","w") as f: f.write(env_example)
with open(".dockerignore","w") as f: f.write(dockerignore)

print("✅ docker-compose.yml written")
print("✅ .env.example written (copy to .env with real values)")
print("✅ .dockerignore written")
print()
print("Local dev workflow:")
print("  cp .env.example .env          # fill in your Azure OpenAI key")
print("  docker-compose up --build     # build + start")
print("  docker-compose logs -f        # tail logs")
print("  docker-compose down           # stop + remove containers")


✅ docker-compose.yml written
✅ .env.example written (copy to .env with real values)
✅ .dockerignore written

Local dev workflow:
  cp .env.example .env          # fill in your Azure OpenAI key
  docker-compose up --build     # build + start
  docker-compose logs -f        # tail logs
  docker-compose down           # stop + remove containers


## Step 4: Inject Secrets via Azure Key Vault (Production Pattern)

In [ ]:
# In production, NEVER use environment variables for secrets.
# Use Azure Key Vault references — the platform fetches the secret, not your code.

import json

keyvault_pattern = {
    "development": {
        "method": "Environment variables via .env file",
        "risk": "Medium — .env can be accidentally committed",
        "setup": "AZURE_OPENAI_KEY=sk-... in .env"
    },
    "staging": {
        "method": "Azure Container Apps secret store",
        "risk": "Low — stored encrypted, not in code",
        "setup": "az containerapp secret set --name app --secret-name oai-key --value $KEY"
    },
    "production": {
        "method": "Azure Key Vault reference",
        "risk": "Lowest — secret never touches app config",
        "setup": "ACA references Key Vault; secret is injected at runtime by Azure"
    }
}

# Key Vault reference syntax for Container Apps
kv_reference = """
# In Container Apps environment variable:
AZURE_OPENAI_KEY=@Microsoft.KeyVault(
    VaultName=kv-llm-prod;
    SecretName=azure-openai-key
)

# This means: Azure fetches the secret from Key Vault at startup.
# The key NEVER appears in your docker image, app config, or logs.
"""

print("Secret Injection Strategy by Environment:")
print("="*55)
for env, info in keyvault_pattern.items():
    print(f"\n  {env.upper()}")
    for k,v in info.items():
        print(f"    {k}: {v}")

print("\nKey Vault Reference Syntax:")
print(kv_reference)

print("Setup commands (Azure Cloud Shell):")
KV_NAME = "kv-llm-prod"
RG      = "rg-llm-demo"
APP     = "llm-api-app"
print(f"  az keyvault create --name {KV_NAME} --resource-group {RG} --sku standard")
print(f"  az keyvault secret set --vault-name {KV_NAME} --name azure-openai-key --value <YOUR_KEY>")
print(f"  az containerapp secret set --name {APP} --resource-group {RG} --secrets oai-key=keyvaultref:{KV_NAME}/azure-openai-key")



Secret Injection Strategy by Environment:

  DEVELOPMENT
    method: Environment variables via .env file
    risk: Medium — .env can be accidentally committed

  STAGING
    method: Azure Container Apps secret store
    risk: Low — stored encrypted, not in code

  PRODUCTION
    method: Azure Key Vault reference
    risk: Lowest — secret never touches app config

Key Vault Reference Syntax: [...]

Setup commands (Azure Cloud Shell):
  az keyvault create --name kv-llm-prod --resource-group rg-llm-demo --sku standard
  az keyvault secret set --vault-name kv-llm-prod --name azure-openai-key --value <YOUR_KEY>


In [ ]:
# Azure Container Registry — Build & Push
# Run commands in Azure Cloud Shell.

ACR = 'acrllmdemo'
RG = 'rg-llm-demo'
IMG = 'llm-api'
TAG = 'v1.0'

steps = [
    ('1. Create ACR (Basic tier)', f'az acr create --name {ACR} --resource-group {RG} --sku Basic --admin-enabled true'),
    ('2. Build image in Azure (no local Docker daemon needed)', f'az acr build --registry {ACR} --image {IMG}:{TAG} --file Dockerfile .'),
    ('3. Scan for CVEs (Defender for Containers)', f'az security pricing create --name ContainerRegistry --tier Standard'),
    ('4. List images', f'az acr repository list --name {ACR} -o table'),
]
print("="*70)
print("  Azure Container Registry — Build & Push")
print("="*70)
for t,c in steps:
    print(f"\n--- {t} ---")
    print(c)
print('ACR acrllmdemo created (Basic, $0.167/day)')
print('Build context uploaded to Azure...')
print('Step 1/14 : FROM python:3.11-slim AS builder')
print('Step 8/14 : FROM python:3.11-slim AS runtime')
print('Step 14/14: CMD [uvicorn ...]')
print('Successfully built and pushed: acrllmdemo.azurecr.io/llm-api:v1.0')
print('Image size: 187MB  (vs 1.4GB single-stage = 87% reduction)')


  Azure Container Registry — Build & Push

--- 1. Create ACR (Basic tier) ---
az acr create --name acrllmdemo --resource-group rg-llm-demo --sku Basic --admin-enabled true

--- 2. Build image in Azure (no local Docker daemon needed) ---
az acr build --registry acrllmdemo --image llm-api:v1.0 --file Dockerfile .

--- 3. Scan for CVEs (Defender for Containers) ---
az security pricing create --name ContainerRegistry --tier Standard

--- 4. List images ---
az acr repository list --name acrllmdemo -o table

[SYNTHETIC DEMO OUTPUT]
ACR acrllmdemo created (Basic, $0.167/day)
Build context uploaded to Azure...
Step 1/14 : FROM python:3.11-slim AS builder
Step 8/14 : FROM python:3.11-slim AS runtime
Step 14/14: CMD [uvicorn ...]
Successfully built and pushed: acrllmdemo.azurecr.io/llm-api:v1.0
Image size: 187MB  (vs 1.4GB single-stage = 87% reduction)


In [ ]:
security_checklist = [
    ("✅", "Non-root USER in Dockerfile"),
    ("✅", "Multi-stage build — build tools not in runtime image"),
    ("✅", "HEALTHCHECK defined for Container Apps/ACA"),
    ("✅", ".dockerignore excludes .env, *.key, .git"),
    ("✅", "Layer order: requirements before code (cache efficiency)"),
    ("✅", "No secrets hardcoded — use .env locally, Key Vault in prod"),
    ("✅", "Pinned package versions in requirements.txt"),
    ("✅", "Minimal base image: python:3.11-slim (not python:3.11)"),
    ("⚠️ ", "Enable ACR vulnerability scanning in production"),
    ("⚠️ ", "Rotate Key Vault secrets every 90 days"),
]
print("🔐 Container Security Checklist")
print("="*50)
for status, item in security_checklist:
    print(f"  {status}  {item}")
print("\n📌 Key Takeaways:")
print("  • Multi-stage builds: 1.4GB → 187MB (87% reduction)")
print("  • Layer ordering = build cache = 5s rebuilds instead of 3min")
print("  • docker-compose for local dev, Key Vault references for production")
print("  • az acr build = build in Azure cloud, no local Docker daemon needed")


🔐 Container Security Checklist
  ✅  Non-root USER in Dockerfile
  ✅  Multi-stage build — build tools not in runtime image
  ✅  HEALTHCHECK defined for Container Apps/ACA
  ✅  .dockerignore excludes .env, *.key, .git
  ✅  Layer order: requirements before code (cache efficiency)
  ✅  No secrets hardcoded — use .env locally, Key Vault in prod
  ✅  Pinned package versions in requirements.txt
  ✅  Minimal base image: python:3.11-slim (not python:3.11)
  ⚠️   Enable ACR vulnerability scanning in production
  ⚠️   Rotate Key Vault secrets every 90 days

📌 Key Takeaways:
  • Multi-stage builds: 1.4GB → 187MB (87% reduction)
  • Layer ordering = build cache = 5s rebuilds instead of 3min
  • docker-compose for local dev, Key Vault references for production
  • az acr build = build in Azure cloud, no local Docker daemon needed
